# Phase 4 — Fine-Tune seal-ocr-rec-v2 on Kaggle GPU

### Hướng dẫn thiết lập trên Kaggle:
1. **Bật Internet**: Tại thanh menu bên phải của Kaggle Notebook: `Settings` → `Internet` → Bật **ON** (bắt buộc để tải PaddleOCR và pretrained weights).
2. **Bật GPU**: `Settings` → `Accelerator` → Chọn **GPU P100** hoặc **GPU T4 x2**.
3. **Tải Dataset lên Kaggle**:
   - Bấm `+ Add Input` ở góc trên bên phải → `Upload Dataset` → Kéo thả file `recognition_dataset_v2.zip` lên (Đặt tên dataset ví dụ `seal-recognition-v2`).
   - Notebook sẽ tự động tìm thấy file zip hoặc thư mục dataset trong `/kaggle/input/`.
4. **Kết quả đầu ra**: Model đã export (`seal-ocr-rec-v2.zip`) sẽ nằm trực tiếp tại thư mục `/kaggle/working/`, bạn có thể tải về máy bằng 1 cú click ở panel bên phải.

In [ ]:
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone

MODEL_VERSION = 'seal-ocr-rec-v2'
SEED = 20260918
RESUME = False

# Set GPU 0 for single card training
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

WORK = Path('/kaggle/working/seal-ocr-phase4')
RUN_DIR = Path('/kaggle/working/runs') / MODEL_VERSION
PADDLE = Path('/kaggle/working/PaddleOCR')

if RUN_DIR.exists() and not RESUME:
    shutil.rmtree(RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True, exist_ok=True)

# Auto-detect dataset in /kaggle/input or /kaggle/working
dataset_zip = None
dataset_dir = None

input_root = Path('/kaggle/input')
if input_root.exists():
    for p in input_root.rglob('recognition_dataset_v2.zip'):
        dataset_zip = p
        break
    if not dataset_zip:
        for p in input_root.rglob('train.txt'):
            dataset_dir = p.parent
            break

if not dataset_zip and not dataset_dir:
    fallback = Path('/kaggle/working/recognition_dataset_v2.zip')
    if fallback.is_file():
        dataset_zip = fallback

if dataset_zip:
    print(f'Found dataset archive: {dataset_zip}')
    with zipfile.ZipFile(dataset_zip) as archive:
        archive.extractall(WORK)
    DATASET = WORK
elif dataset_dir:
    print(f'Found unpacked dataset directory: {dataset_dir}')
    DATASET = dataset_dir
else:
    raise FileNotFoundError('Could not find recognition_dataset_v2.zip in /kaggle/input/. Please upload it via Add Input!')

print({'runDir': str(RUN_DIR), 'dataset': str(DATASET)})

In [ ]:
# Validate dataset integrity
metadata_file = DATASET / 'metadata.json'
if metadata_file.is_file():
    meta = json.loads(metadata_file.read_text())
    print('Dataset metadata:', meta)
    assert meta.get('datasetVersion') == 'seal-recognition-v2', 'Unexpected dataset version'
    assert meta.get('trainSamples', 0) >= 4000, f'Expected >= 4000 train samples, got {meta.get("trainSamples")}'
    assert meta.get('valSamples', 0) >= 400, f'Expected >= 400 val samples, got {meta.get("valSamples")}'

from PIL import Image
def read_labels(filename):
    rows = {}
    for line_num, line in enumerate((DATASET / filename).read_text(encoding='utf-8').splitlines(), 1):
        if not line.strip():
            continue
        parts = line.split('\t', 1)
        if len(parts) != 2:
            raise ValueError(f'Malformed line {line_num} in {filename}: {line}')
        crop_rel, label = parts
        crop_path = DATASET / crop_rel
        if not crop_path.is_file():
            raise FileNotFoundError(f'Missing image file: {crop_path}')
        if not re.fullmatch(r'[A-Z0-9]{5,20}', label):
            raise ValueError(f'Invalid label characters in {filename}:{line_num}: {label}')
        rows[crop_rel] = label
    return rows

train_rows = read_labels('train.txt')
val_rows = read_labels('val.txt')
assert not (set(train_rows) & set(val_rows)), 'Leakage detected between train and val splits!'
print(f'Verified dataset: {len(train_rows)} train samples, {len(val_rows)} val samples.')

In [ ]:
# Clone PaddleOCR source & install PaddlePaddle GPU wheel
if not PADDLE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'v3.7.0',
                    'https://github.com/PaddlePaddle/PaddleOCR.git', str(PADDLE)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'paddlepaddle', 'paddlepaddle-gpu'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'paddlepaddle-gpu==3.2.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PADDLE / 'requirements.txt'), 'pyyaml'], check=True)

import paddle
assert paddle.device.is_compiled_with_cuda(), 'Paddle GPU wheel is not compiled with CUDA!'
paddle.utils.run_check()
print({'paddle': paddle.__version__, 'device': paddle.device.get_device()})

In [ ]:
# Configure training recipe from official PP-OCRv6_medium_rec
import urllib.request, yaml
CONFIG_SOURCE = 'https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/v3.7.0/configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml'
source_config = WORK / 'official_PP-OCRv6_medium_rec.yml'
urllib.request.urlretrieve(CONFIG_SOURCE, source_config)
config = yaml.safe_load(source_config.read_text())

config['Global'].update({
    'epoch_num': 60,
    'seed': SEED,
    'save_model_dir': str(RUN_DIR / 'checkpoints'),
    'save_epoch_step': 10,
    'eval_batch_step': [0, 100],
    'distributed': False,
    'use_visualdl': False,
    'max_text_length': 25
})
config['Optimizer']['lr'].update({'learning_rate': 0.00005, 'warmup_epoch': 2})

# Dataset configuration
config['Train']['dataset']['data_dir'] = str(DATASET)
config['Train']['dataset']['label_file_list'] = [str(DATASET / 'train.txt')]
config['Train']['dataset']['transforms'] = [
    {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
    {'RecAug': {'tia_prob': 0.10, 'crop_prob': 0.0, 'reverse_prob': 0.0, 'noise_prob': 0.15,
                'jitter_prob': 0.05, 'blur_prob': 0.10, 'hsv_aug_prob': 0.15}},
    {'MultiLabelEncode': {'gtc_encode': 'NRTRLabelEncode'}},
    {'KeepKeys': {'keep_keys': ['image', 'label_ctc', 'label_gtc', 'length', 'valid_ratio']}},
]
config['Train']['loader'].update({'batch_size_per_card': 32, 'drop_last': False, 'num_workers': 2})

config['Eval']['dataset']['data_dir'] = str(DATASET)
config['Eval']['dataset']['label_file_list'] = [str(DATASET / 'val.txt')]
config['Eval']['loader'].update({'batch_size_per_card': 64, 'num_workers': 2})

CONFIG = RUN_DIR / 'config.yml'
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))

PRETRAIN = RUN_DIR / 'PP-OCRv6_medium_rec_pretrained.pdparams'
if not PRETRAIN.exists():
    print('Downloading pretrained checkpoint...')
    urllib.request.urlretrieve('https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_medium_rec_pretrained.pdparams', PRETRAIN)

print('Config and pretrained checkpoint ready!')

In [ ]:
# Execute Fine-Tuning
CHECKPOINTS = RUN_DIR / 'checkpoints'
overrides = [
    f'Global.save_model_dir={CHECKPOINTS}',
    f'Global.pretrained_model={PRETRAIN}',
    'Global.use_amp=True',
    'Global.scale_loss=1024.0',
    'Global.use_dynamic_loss_scaling=True'
]

print('Starting fine-tuning for 60 epochs...')
train_process = subprocess.run([sys.executable, 'tools/train.py', '-c', str(CONFIG), '-o', *overrides],
                               cwd=PADDLE, text=True)
if train_process.returncode != 0:
    raise RuntimeError('Training failed! Check output log above.')

print('Training successfully finished!')

In [ ]:
# Export best checkpoint to Inference model format
BEST = CHECKPOINTS / 'best_accuracy'
INFERENCE = RUN_DIR / 'inference'
INFERENCE.mkdir(parents=True, exist_ok=True)

print('Exporting best model to inference format...')
export_process = subprocess.run([
    sys.executable, 'tools/export_model.py', '-c', str(CONFIG),
    '-o', f'Global.pretrained_model={BEST}', f'Global.save_inference_dir={INFERENCE}'
], cwd=PADDLE, text=True)
if export_process.returncode != 0:
    raise RuntimeError('Export failed!')

for name in ('inference.json', 'inference.pdiparams', 'inference.yml'):
    if not (INFERENCE / name).is_file():
        raise FileNotFoundError(f'Missing exported file: {name}')

# Create zip directly in /kaggle/working/ for instant download
OUT_ZIP = Path('/kaggle/working') / 'seal-ocr-rec-v2'
shutil.make_archive(str(OUT_ZIP), 'zip', root_dir=INFERENCE)

print(f'\nSUCCESS! Model exported and archived at: /kaggle/working/seal-ocr-rec-v2.zip')
print('You can now download seal-ocr-rec-v2.zip directly from the Output section in the right panel!')